# 02 · Entrenamiento YOLOv8

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elianpinzas/vision_por_computadora_2_tp_final/blob/main/notebooks/02_train_yolov8.ipynb)

**Responsable:** _(a definir)_

**Objetivo:** Fine-tuning de YOLOv8 sobre Wildfire Smoke con la configuración común. Detector CNN de una etapa.

**Usa:** Wildfire Smoke (versión de `config.yaml`). Requiere el secret `ROBOFLOW_API_KEY`.

**Entrega:** `best.pt`, `results.csv` y `args.yaml`, subidos como Release del repo.

## 0. Preparación del entorno
Estas celdas son iguales en todos los notebooks. Clonan el repo, instalan las dependencias y cargan la configuración común. Todo lo que se descarga queda en el disco de la máquina virtual de Colab y se borra al cerrar la sesión.

In [ ]:
# Clonar el repo (en Colab el notebook se abre solo, sin el resto de los archivos)
import os
if not os.path.exists("vision_por_computadora_2_tp_final"):
    !git clone https://github.com/elianpinzas/vision_por_computadora_2_tp_final.git
%cd vision_por_computadora_2_tp_final
!pip install -q -r requirements.txt

In [ ]:
# Cargar la configuración común del grupo
from src.utils import load_config, set_seed
from src import datos
cfg = load_config("config.yaml")
set_seed(cfg["seed"])
cfg

## 1. Descargar el dataset
Se baja al disco de la sesión con la función común.

In [ ]:
datos.descargar_wildfire(cfg)

## 2. Entrenar
Todos los hiperparámetros salen de `config.yaml`, nada se escribe a mano acá, para que la comparación sea justa.

In [ ]:
from ultralytics import YOLO
t = cfg["train"]
model = YOLO(cfg["models"]["yolov8"])
# TODO: descomentar cuando config.yaml esté acordado
# model.train(
#     data=cfg["datasets"]["wildfire_smoke"]["data_yaml"],
#     imgsz=t["imgsz"], epochs=t["epochs"], batch=t["batch"],
#     optimizer=t["optimizer"], lr0=t["lr0"], patience=t["patience"],
#     seed=cfg["seed"], deterministic=True,
#     project=cfg["paths"]["runs"], name="yolov8", exist_ok=True,
# )

## 3. Validar en el test de Wildfire Smoke
Métricas sobre el split de test, con el mejor peso.

In [ ]:
# TODO: model.val(data=..., split='test')

## 4. Guardar los pesos (antes de cerrar la sesión)
Todo lo que está en `runs/` se borra al cerrar la sesión. Descargar estos tres archivos y subirlos al repo como Release:

1. En GitHub: *Releases → Draft a new release*, con el tag de `config.yaml` (`release.tag`). Si la release ya existe, editarla.
2. Adjuntar los archivos con estos nombres: `yolov8_best.pt`, `yolov8_results.csv` y `yolov8_args.yaml`.
3. Publicar. El notebook de evaluación los descarga solo desde ahí.

In [ ]:
import shutil
from google.colab import files
run = f"{cfg['paths']['runs']}/yolov8"
for origen, destino in [("weights/best.pt", "yolov8_best.pt"), ("results.csv", "yolov8_results.csv"), ("args.yaml", "yolov8_args.yaml")]:
    shutil.copy(f"{run}/{origen}", destino)
    files.download(destino)